# 09 — What does radiation do to the SPP1-TAM state? (single-cell)

> **ANSWER: nothing that this dataset can detect — and the reason is not SPP1-TAM-specific.**
> Radiation leaves **no reproducible transcriptional signature in any cell compartment** here,
> while Cy + CAR-T leaves a strong one in every compartment through the identical pipeline.
> Specifically: **SPP1-TAMs do not increase their fatty-acid/lipid program after RT.**

This notebook exists because 06 and 07 answered *"does RT change how many SPP1-TAMs there are"*
(no — prevalence falls slightly with RT in both CAR-T backgrounds) and 08 asks the **spatial**
question. Neither asked the cell-intrinsic one:

> **Among SPP1-TAMs, does adding RT shift them toward fatty-acid processing?**

That is a single-cell question — dissociated, so free of the segmentation spillover that killed
the spatial Spp1 call in 01 §5 — and it is the mouse counterpart of the human "8 Gy vs sham"
OXPHOS/fatty-acid result in the proposal.

### The design constraint, restated
There is **no RT-alone arm**: every irradiated mouse also received Cy + CAR-T. So every contrast
reads as *"adding RT on top of Cy + CAR-T"*, never *"RT vs untreated"*. Unlike the spatial object,
**NoTx is usable here** (2,059 macrophages across 6 samples), which is what makes the positive
control in §7 possible.

| CAR-T background | irradiated arm | matched non-RT baseline | n |
|---|---|---|---|
| PSCA  | `RTCyPSCA`  | `CyPSCA`  | 6 vs 6 tumours (3 vs 3 mice) |
| TAG72 | `RTCyTAG72` | `CyTAG72` | 6 vs 5 tumours (3 vs 3 mice) |

Two independent backgrounds → **the evidence is reproducibility across them**, not a p-value.
`Tu1` is the irradiated tumour and `Tu2` the abscopal one **in the same mouse**, so `Tu1 − Tu2`
inside an RT arm is a within-animal radiation contrast — the strongest the design allows (§10).

> **Arm-name trap:** spatial spells these `CyT72`/`RTCyT72`; the single-cell object spells them
> `CyTAG72`/`RTCyTAG72`. Mismatching them silently drops every CAR-T-specificity contrast.

### Why this notebook is mostly controls
Three things would each produce a *positive-looking* fatty-acid result, and all three are live here:

1. **A per-sample global offset.** Pseudobulk mean-log-normalised expression carries a library-
   complexity term — a sample with more genes detected spreads the same CP10K budget thinner and
   every gene shifts. §5 shows this offset is the *largest* thing in the naive between-arm contrast.
   → removed by an **internal reference** (TAM − non-myeloid of the same sample) and by the
   within-mouse pairing.
2. **Circularity.** The 50-gene `SPP1_TAM_sig` contains `Apoe`, `Abca1`, `Fabp5`, `Slc27a1`. Selecting
   cells by it and then measuring lipid pathways is partly self-fulfilling. → signature genes are
   **removed from every pathway set**, the population is defined **four independent ways**, and the
   key gene-level tests are repeated on `M2_like_Mac`, which is defined without the signature.
3. **Mouse-level noise mistaken for an arm effect.** n = 3 mice per arm, and the gene-level logFC of
   an arm contrast is dominated by which 3 mice landed in it. → every correlation gets a
   **label-permutation null** (§8), because 8,000 genes are not 8,000 independent tests.

**Prereq:** `01c` §5 (`sc_spp1_tam_signature.csv`). Helpers are inline — the HPC kernel cannot see
local `.py` files.

## 1. Config

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, scanpy as sc, anndata as ad, h5py
import scipy.sparse as sp
from scipy import stats
from pathlib import Path

# Runs on the HPC kernel or on a local copy — whichever resolves first.
CANDIDATES = [
    Path("/coh_labs/yunroseli/Jona/CAR-T"),
    Path("/Users/janzules/Roselab/Spatial/CAR_T"),
]
PROJ = next(p for p in CANDIDATES if p.exists())
if PROJ.as_posix().startswith("/coh_labs"):
    SC_REF  = PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad"
    MAC_REF = PROJ/"data/zarr/fullDataset/annotating_references/mac_annotation_v5.h5ad"
    GMT_DIR = PROJ/"data/references/pathways"
else:
    SC_REF  = PROJ/"data/sc-reference/annotation_v4.h5ad"
    MAC_REF = PROJ/"data/sc-reference/mac_annotation_v5.h5ad"
    GMT_DIR = PROJ/"data/references/pathways"
SIG_CSV = PROJ/"results/spp1_analysis/sc_spp1_tam_signature.csv"
OUT     = PROJ/"results/spp1_analysis/sc_rt_effect"; OUT.mkdir(parents=True, exist_ok=True)

MAC_LABELS = ["M1_like_Mac", "M2_like_Mac", "Intermediate_Mac"]
MYELOID    = MAC_LABELS + ["Classical_Mono","Nonclassical_Mono","cDC","pDC","N1_like_Neu","N2_like_Neu"]
BG         = {"PSCA": ("RTCyPSCA","CyPSCA"), "TAG72": ("RTCyTAG72","CyTAG72")}
MIN_CELLS  = 20        # per-sample floor for a usable pseudobulk
DET_FLOOR  = 0.10      # gene must be detected in >10% of the cells being tested
rng = np.random.default_rng(0)
pd.set_option("display.width", 220, "display.max_columns", 60)
print(f"project root : {PROJ}\nreference    : {SC_REF}\noutputs      : {OUT}")

project root : /Users/janzules/Roselab/Spatial/CAR_T
reference    : /Users/janzules/Roselab/Spatial/CAR_T/data/sc-reference/annotation_v4.h5ad
outputs      : /Users/janzules/Roselab/Spatial/CAR_T/results/spp1_analysis/sc_rt_effect


## 2. Load, and rebuild the design from `file_name`

`.X` is checked rather than assumed — it is log1p-normalised in this object, and `layers['counts']`
is raw. Reading it wrong would silently corrupt every score below. The counts layer is **not**
loaded (it is 3 GB and nothing here needs it).

In [2]:
with h5py.File(SC_REF, "r") as f:
    obs = ad.io.read_elem(f["obs"]); var = ad.io.read_elem(f["var"]); X = ad.io.read_elem(f["X"])
adata = ad.AnnData(X=X, obs=obs, var=var); del X

mx = adata.X[:2000].max()
assert mx < 50, f".X max = {mx} — these are raw counts, not log-normalised. ABORT."
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes | .X max {mx:.2f} -> log-normalised, used as-is")

o = adata.obs
o["cond"]   = o["condition"].astype(str)
o["sample"] = o["file_name"].astype(str)
o["tu"]     = o["sample"].str.extract(r"_(Tu\d)_")[0].values     # Tu1 = irradiated, Tu2 = abscopal
o["mouse"]  = o["sample"].str.replace(r"_Tu\d_", "_", regex=True)
o["ct"]     = o["cell_type_lvl2"].astype(str)
o["is_mac"] = o["ct"].isin(MAC_LABELS)
o["is_M2"]  = o["ct"] == "M2_like_Mac"
o["is_nonmyeloid"] = ~o["ct"].isin(MYELOID)

print(f"\nmacrophages            : {o.is_mac.sum():,}")
print(f"non-myeloid (reference): {o.is_nonmyeloid.sum():,}")
print(f"\nsamples per arm x site:\n{o.groupby(['cond','tu'], observed=True)['sample'].nunique().unstack()}")

93,548 cells x 22,949 genes | .X max 7.94 -> log-normalised, used as-is

macrophages            : 48,475
non-myeloid (reference): 9,137

samples per arm x site:
tu         Tu1  Tu2
cond               
CyPSCA       3    3
CyTAG72      2    3
NoTx         3    3
RTCyPSCA     3    3
RTCyTAG72    3    3


## 3. The SPP1-TAM population, defined four independent ways

No single definition is trusted. If the RT answer is the same under all four, it does not depend
on how the population was drawn — and if it differs, the definition *is* the finding.

| definition | how | what it controls for |
|---|---|---|
| `TAM_global` | top quartile of `SPP1_TAM_sig`, one cut across all macs | 07's convention; prevalence-aware |
| `TAM_within` | top quartile **per sample** | prevalence-blind — pure cell state, no arm composition |
| `TAM_alice`  | `cell_type_lvl4 == Spp1_pos_mac` from `mac_annotation_v5` | **independent of our signature entirely** |
| `is_M2`      | `M2_like_Mac` | the host compartment — 07 showed the SPP1-TAM pool is ~100% M2 |

In [3]:
sig = [g for g in pd.read_csv(SIG_CSV)["gene"] if g in adata.var_names]
print(f"SPP1-TAM signature: {len(sig)}/50 genes present")
sc.tl.score_genes(adata, sig, score_name="SPP1_TAM_sig", ctrl_size=50, random_state=0)

mac, s = o["is_mac"].values, adata.obs["SPP1_TAM_sig"].values
o["TAM_global"] = mac & (s >= np.quantile(s[mac], 0.75))

d2 = np.zeros(adata.n_obs, bool)
for smp in o["sample"].unique():
    m = mac & (o["sample"].values == smp)
    if m.sum() >= MIN_CELLS: d2[m] = s[m] >= np.quantile(s[m], 0.75)
o["TAM_within"] = d2

with h5py.File(MAC_REF, "r") as f: m_obs = ad.io.read_elem(f["obs"])
lvl4 = m_obs["cell_type_lvl4"].astype(str)
o["TAM_alice"] = o.index.map(lvl4.reindex(o.index)).fillna("NA").values == "Spp1_pos_mac"

print("\n--- population definitions ---")
for d in ["TAM_global","TAM_within","TAM_alice","is_M2"]:
    print(f"  {d:<12s} {o[d].sum():>7,}  ({100*o[d].sum()/mac.sum():5.1f}% of macrophages)")
print("\n  Jaccard vs TAM_global:")
for d in ["TAM_within","TAM_alice","is_M2"]:
    a, b = o["TAM_global"].values, o[d].values
    print(f"    {d:<12s} {(a&b).sum()/(a|b).sum():.3f}")

SPP1-TAM signature: 50/50 genes present



--- population definitions ---
  TAM_global    12,119  ( 25.0% of macrophages)
  TAM_within    12,131  ( 25.0% of macrophages)
  TAM_alice     29,582  ( 61.0% of macrophages)
  is_M2         23,538  ( 48.6% of macrophages)

  Jaccard vs TAM_global:
    TAM_within   0.736
    TAM_alice    0.343
    is_M2        0.514


## 4. Pathway panel

Three tiers, and the **control tier is the whole argument**. If IFN-γ / TNF-α / inflammatory move
as much as the metabolic tier, RT changed the sample globally and nothing is lipid-specific.
The **RT-damage tier** (p53, DNA repair, apoptosis, ROS) is the positive control for the beam
itself having left a fingerprint.

**Signature genes are stripped from every set** before scoring — the population is defined by those
genes, so leaving them in would let the definition leak into the readout.

In [4]:
METABOLIC = {
  "FA_uptake_import":"GOBP_LONG_CHAIN_FATTY_ACID_IMPORT_INTO_CELL", "FA_transport":"GOBP_FATTY_ACID_TRANSPORT",
  "FA_metabolism_H":"HALLMARK_FATTY_ACID_METABOLISM", "FA_metabolism_R":"REACTOME_FATTY_ACID_METABOLISM",
  "FA_beta_oxidation":"GOBP_FATTY_ACID_BETA_OXIDATION", "FA_catabolism":"GOBP_FATTY_ACID_CATABOLIC_PROCESS",
  "lipid_oxidation":"GOBP_LIPID_OXIDATION", "FA_biosynthesis":"GOBP_FATTY_ACID_BIOSYNTHETIC_PROCESS",
  "TG_catabolism":"GOBP_TRIGLYCERIDE_CATABOLIC_PROCESS", "lipid_droplet":"GOBP_LIPID_DROPLET_ORGANIZATION",
  "lipophagy":"GOBP_LIPOPHAGY", "foam_cell":"GOBP_MACROPHAGE_DERIVED_FOAM_CELL_DIFFERENTIATION",
  "oxLDL_response":"GOBP_CELLULAR_RESPONSE_TO_OXIDISED_LOW_DENSITY_LIPOPROTEIN_PARTICLE_STIMULUS",
  "chol_efflux":"GOBP_CHOLESTEROL_EFFLUX", "chol_storage":"GOBP_CHOLESTEROL_STORAGE",
  "chol_homeostasis":"HALLMARK_CHOLESTEROL_HOMEOSTASIS", "adipogenesis":"HALLMARK_ADIPOGENESIS",
  "peroxisome":"HALLMARK_PEROXISOME", "lysosomal_transport":"GOBP_LYSOSOMAL_TRANSPORT",
  "OXPHOS":"HALLMARK_OXIDATIVE_PHOSPHORYLATION", "glycolysis":"HALLMARK_GLYCOLYSIS",
  "mTORC1":"HALLMARK_MTORC1_SIGNALING", "hypoxia":"HALLMARK_HYPOXIA"}
CONTROL = {"IFN_gamma":"HALLMARK_INTERFERON_GAMMA_RESPONSE","IFN_alpha":"HALLMARK_INTERFERON_ALPHA_RESPONSE",
  "inflammatory":"HALLMARK_INFLAMMATORY_RESPONSE","TNFA_NFKB":"HALLMARK_TNFA_SIGNALING_VIA_NFKB",
  "complement":"HALLMARK_COMPLEMENT","MYC_targets":"HALLMARK_MYC_TARGETS_V1","G2M":"HALLMARK_G2M_CHECKPOINT"}
RT_DAMAGE = {"p53_pathway":"HALLMARK_P53_PATHWAY","DNA_repair":"HALLMARK_DNA_REPAIR",
  "apoptosis":"HALLMARK_APOPTOSIS","ROS_pathway":"HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY"}
ALL_SETS = {**METABOLIC, **CONTROL, **RT_DAMAGE}
TIER = {**{k:"metabolic" for k in METABOLIC}, **{k:"control" for k in CONTROL},
        **{k:"rt_damage" for k in RT_DAMAGE}}
FA = [k for k in METABOLIC if k not in ("glycolysis","mTORC1","hypoxia")]   # the 20 lipid/oxidative sets

def load_gmt(d):
    out = {}
    for f in sorted(Path(d).glob("*.gmt")):
        for line in open(f):
            p = line.rstrip("\n").split("\t")
            if len(p) > 2 and p[0] not in out: out[p[0]] = [g for g in p[2:] if g]
    return out

gmt, sig_set, meta = load_gmt(GMT_DIR), set(sig), []
for short, full in ALL_SETS.items():
    present = [g for g in gmt[full] if g in adata.var_names]
    kept    = [g for g in present if g not in sig_set]
    sc.tl.score_genes(adata, kept, score_name=short, ctrl_size=50, random_state=0)
    meta.append(dict(short=short, full=full, tier=TIER[short], n_in_data=len(present),
                     n_dropped_sig=len(present)-len(kept), n_scored=len(kept)))
pw = pd.DataFrame(meta); pw.to_csv(OUT/"pathway_meta.csv", index=False)
SETS = pw.short.tolist()
adata.obs["FA_composite"] = adata.obs[FA].mean(axis=1)
print(f"scored {len(pw)} sets ({(pw.tier=='metabolic').sum()} metabolic / "
      f"{(pw.tier=='control').sum()} control / {(pw.tier=='rt_damage').sum()} RT-damage)")
print(f"signature genes removed from sets: {pw.n_dropped_sig.sum()} total, max {pw.n_dropped_sig.max()} in one set")

scored 34 sets (23 metabolic / 7 control / 4 RT-damage)
signature genes removed from sets: 30 total, max 3 in one set


## 5. §A — The naive between-arm contrast, and why it cannot be believed

Per-sample mean score inside the SPP1-TAM pool, RT arm minus CAR-T-matched non-RT arm,
computed within each tumour site and averaged.

**Watch the control tier.** `TNFA_NFKB` is the single largest mover in both backgrounds and it
moves in **opposite directions** (−0.05 PSCA, +0.11 TAG72). A control pathway outrunning the whole
metabolic tier, with opposite signs, is the signature of a per-sample global offset — not biology.
Everything in §6 onward exists to remove it.

In [5]:
def per_sample(mask, cols=None):
    cols = cols or SETS
    d = adata.obs.loc[mask]
    g = d.groupby("sample", observed=True)
    out = g[cols].mean(); out["n_cells"] = g.size()
    return out.join(d.groupby("sample", observed=True)[["cond","tu","mouse"]].first())

def between_arm(mask, popname):
    ps = per_sample(mask); ps = ps[ps.n_cells >= MIN_CELLS]
    rows = []
    for bg, (rt_arm, ct_arm) in BG.items():
        rt, ct = ps[ps.cond==rt_arm], ps[ps.cond==ct_arm]
        for s in SETS:
            # tumour-site-matched: delta within Tu1 and within Tu2, then averaged
            deltas = [rt.loc[rt.tu==t, s].mean() - ct.loc[ct.tu==t, s].mean()
                      for t in ["Tu1","Tu2"] if (rt.tu==t).any() and (ct.tu==t).any()]
            rows.append(dict(population=popname, background=bg, pathway=s, tier=TIER[s],
                             delta=np.mean(deltas)))
    w = (pd.DataFrame(rows).pivot_table(index=["pathway","tier"], columns="background", values="delta")
           .rename(columns={"PSCA":"d_PSCA","TAG72":"d_TAG72"}).reset_index())
    w["mean_d"] = w[["d_PSCA","d_TAG72"]].mean(axis=1)
    w["agree"]  = np.where(np.sign(w.d_PSCA)==np.sign(w.d_TAG72),
                           np.where(w.d_PSCA>0,"UP both","DOWN both"), "disagree")
    return w.sort_values("mean_d", ascending=False)

A = between_arm(adata.obs.TAM_global.values, "TAM_global")
A.to_csv(OUT/"contrastA_between.csv", index=False)
for tier in ["metabolic","rt_damage","control"]:
    print(f"\n--- {tier} tier — RT minus matched non-RT, inside SPP1-TAMs ---")
    print(A[A.tier==tier][["pathway","d_PSCA","d_TAG72","mean_d","agree"]]
            .to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
print(f"\nBIGGEST ABSOLUTE MOVER OVERALL: "
      f"{A.reindex(A[['d_PSCA','d_TAG72']].abs().max(axis=1).sort_values(ascending=False).index).iloc[0].pathway}"
      f"  <- if this is in the CONTROL tier, the contrast is measuring a global offset.")


--- metabolic tier — RT minus matched non-RT, inside SPP1-TAMs ---
            pathway  d_PSCA  d_TAG72  mean_d     agree
            hypoxia -0.0107  +0.0419 +0.0156  disagree
          lipophagy +0.0103  +0.0167 +0.0135   UP both
   chol_homeostasis -0.0000  +0.0262 +0.0131  disagree
   FA_uptake_import -0.0007  +0.0242 +0.0117  disagree
          foam_cell +0.0025  +0.0104 +0.0065   UP both
         glycolysis +0.0035  +0.0066 +0.0050   UP both
             mTORC1 -0.0072  +0.0172 +0.0050  disagree
       FA_transport -0.0001  +0.0046 +0.0022  disagree
       chol_storage +0.0211  -0.0175 +0.0018  disagree
         peroxisome +0.0029  +0.0006 +0.0017   UP both
lysosomal_transport +0.0038  -0.0021 +0.0009  disagree
    FA_biosynthesis +0.0003  -0.0014 -0.0005  disagree
      lipid_droplet -0.0032  +0.0021 -0.0005  disagree
  FA_beta_oxidation -0.0030  +0.0013 -0.0008  disagree
      FA_catabolism -0.0034  +0.0008 -0.0013  disagree
    FA_metabolism_H -0.0023  -0.0008 -0.0015 DOWN bo

## 6. §B — Two ways to remove the offset, and what survives

* **Internal reference** — `score(SPP1-TAM) − score(non-myeloid cells of the same sample)`.
  Cancels anything that scales a whole sample (depth, complexity, dissociation, batch).
* **Within-mouse pairing** — `Tu1 − Tu2`, same animal, same batch, same systemic therapy. The
  only difference is the beam. The identical quantity in the **non-RT arms** is the control for
  the fact that the two tumour sites differ at baseline, which turns this into a
  difference-in-differences.

In [6]:
tam = per_sample(adata.obs.TAM_global.values)
ref = per_sample(adata.obs.is_nonmyeloid.values)
keep = tam.index[tam.n_cells>=MIN_CELLS].intersection(ref.index[ref.n_cells>=MIN_CELLS])
rel = (tam.loc[keep, SETS] - ref.loc[keep, SETS]).join(tam.loc[keep, ["cond","tu","mouse"]])
rel["FA_composite"] = rel[FA].mean(axis=1)
print(f"usable samples: {len(keep)}/29")

print("\n--- does the internal reference remove the offset? ---")
for lbl, t in [("RAW  TAM score", tam.loc[keep]), ("REL  TAM - non-myeloid", rel)]:
    print(f"\n  {lbl}")
    print(t.groupby("cond", observed=True)[["TNFA_NFKB","IFN_gamma","FA_metabolism_H","OXPHOS"]]
           .mean().to_string(float_format=lambda x: f"{x:+.4f}"))

def paired_deltas(tbl, arm, cols):
    a = tbl[tbl.cond==arm].reset_index()
    w = a.pivot_table(index="mouse", columns="tu", values=cols)
    return pd.DataFrame({c: (w[c]["Tu1"]-w[c]["Tu2"]) for c in cols
                         if "Tu1" in w[c].columns and "Tu2" in w[c].columns})

rows = []
for arm in ["RTCyPSCA","RTCyTAG72","CyPSCA","CyTAG72","NoTx"]:
    d = paired_deltas(rel, arm, SETS)
    for s in SETS:
        if s in d and d[s].notna().sum() >= 2:
            rows.append(dict(arm=arm, pathway=s, tier=TIER[s], delta=d[s].mean()))
W = pd.DataFrame(rows).pivot_table(index=["pathway","tier"], columns="arm", values="delta")
W["RT_mean"]     = W[["RTCyPSCA","RTCyTAG72"]].mean(axis=1)
W["noRT_mean"]   = W[["CyPSCA","CyTAG72"]].mean(axis=1)
W["RT_specific"] = W.RT_mean - W.noRT_mean
W = W.reset_index(); W.to_csv(OUT/"within_mouse_DiD.csv", index=False)

print("\n\n--- WITHIN-MOUSE difference-in-differences: (Tu1-Tu2 | RT) - (Tu1-Tu2 | no RT) ---")
for tier in ["metabolic","rt_damage","control"]:
    print(f"\n  {tier} tier:")
    print(W[W.tier==tier].sort_values("RT_specific", ascending=False)
           [["pathway","RTCyPSCA","RTCyTAG72","RT_mean","noRT_mean","RT_specific"]]
           .to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
print("\n  READ: the metabolic tier does not stand above the control tier. TNFA_NFKB, IFN_alpha and")
print("        IFN_gamma have RT-specific deltas as large as anything in the lipid panel.")

usable samples: 29/29

--- does the internal reference remove the offset? ---

  RAW  TAM score
           TNFA_NFKB  IFN_gamma  FA_metabolism_H  OXPHOS
cond                                                    
CyPSCA       +0.3260    +0.3840          +0.0509 +0.1410
CyTAG72      +0.2121    +0.2801          +0.0499 +0.1461
NoTx         +0.2607    +0.1110          +0.0771 +0.1876
RTCyPSCA     +0.2746    +0.3765          +0.0486 +0.1446
RTCyTAG72    +0.3235    +0.2831          +0.0481 +0.1360

  REL  TAM - non-myeloid
           TNFA_NFKB  IFN_gamma  FA_metabolism_H  OXPHOS
cond                                                    
CyPSCA       +0.2417    +0.2922          -0.0222 -0.0383
CyTAG72      +0.1471    +0.2090          -0.0085 +0.0228
NoTx         +0.2153    +0.1253          -0.0160 -0.0418
RTCyPSCA     +0.1981    +0.2757          -0.0194 -0.0241
RTCyTAG72    +0.2168    +0.1871          -0.0116 -0.0064


--- WITHIN-MOUSE difference-in-differences: (Tu1-Tu2 | RT) - (Tu1-Tu2 | no RT)

## 7. §C — Does the fatty-acid readout have power at all?

**A null is worthless until the same measurement is shown to detect something it is known to
contain.** Three positive controls, in these exact cells, with this exact scoring:

1. **Spp1-high vs Spp1-low macrophages** — 01c's result, the SPP1-TAM state axis itself.
2. **M2 vs M1 macrophages** — 07 showed the program is ~100% M2.
3. **NoTx vs treated** — 07 showed prevalence and state both move.

In [7]:
mac_v = adata.obs.is_mac.values
sig_v = adata.obs.SPP1_TAM_sig.values
lo    = mac_v & (sig_v <= np.quantile(sig_v[mac_v], 0.25))
fa    = adata.obs.FA_composite.values

def eff(a, b, lbl):
    x, y = fa[a], fa[b]
    d = (x.mean()-y.mean())/np.sqrt((x.var()+y.var())/2)
    print(f"  {lbl:<44s} {x.mean():+.4f} vs {y.mean():+.4f}  delta {x.mean()-y.mean():+.4f}  "
          f"Cohen d {d:+.2f}  (n {a.sum():,} vs {b.sum():,})")
    return x.mean()-y.mean()

print("FA composite, cell level:")
d_state = eff(adata.obs.TAM_global.values, lo, "SPP1-TAM-high vs Spp1-low macs (01c)")
eff(adata.obs.is_M2.values, (adata.obs.ct=="M1_like_Mac").values, "M2_like_Mac vs M1_like_Mac")
eff(mac_v & (adata.obs.cond=="NoTx").values, mac_v & (adata.obs.cond!="NoTx").values,
    "NoTx macs vs treated macs")

lo_s = adata.obs[lo].groupby("sample", observed=True)["FA_composite"].mean()
vs   = (tam.loc[keep,"FA_composite"] if "FA_composite" in tam else
        adata.obs[adata.obs.TAM_global].groupby("sample", observed=True)["FA_composite"].mean()) - lo_s
print(f"\nSample level — TAM-high minus Spp1-low WITHIN each sample:")
print(f"  {vs.mean():+.4f} +/- {vs.std():.4f}, positive in {int((vs>0).sum())}/{len(vs)} samples")

sd_within = rel.groupby("cond", observed=True)["FA_composite"].std().mean()
mde = 2.78 * sd_within * np.sqrt(2/6)
print(f"\nMINIMUM DETECTABLE EFFECT on the internal-reference FA composite:")
print(f"  within-arm sd {sd_within:.4f} | n=6 tumours/side | MDE (80% power, a=0.05) = {mde:.4f}")
print(f"  the SPP1-TAM state effect is {vs.mean():+.4f}  ->  MDE is {100*mde/vs.mean():.0f}% of it")
print(f"  => an RT effect even a THIRD the size of the state axis could not have been missed.")

FA composite, cell level:
  SPP1-TAM-high vs Spp1-low macs (01c)         +0.0692 vs +0.0434  delta +0.0258  Cohen d +1.24  (n 12,119 vs 12,119)
  M2_like_Mac vs M1_like_Mac                   +0.0653 vs +0.0463  delta +0.0190  Cohen d +0.91  (n 23,538 vs 15,716)
  NoTx macs vs treated macs                    +0.0637 vs +0.0553  delta +0.0084  Cohen d +0.35  (n 2,059 vs 46,416)

Sample level — TAM-high minus Spp1-low WITHIN each sample:
  +0.0259 +/- 0.0048, positive in 29/29 samples

MINIMUM DETECTABLE EFFECT on the internal-reference FA composite:
  within-arm sd 0.0058 | n=6 tumours/side | MDE (80% power, a=0.05) = 0.0093
  the SPP1-TAM state effect is +0.0259  ->  MDE is 36% of it
  => an RT effect even a THIRD the size of the state axis could not have been missed.


## 8. §D — The decisive test: is the RT effect reproducible across the two backgrounds?

Module scores average over hundreds of genes and can hide a focused shift, so this moves to
gene level — and to the one statistic that n = 3 mice per arm can actually support.

Rather than asking "is any gene significant" (it is not — 0 at q<0.05 in either background),
ask whether the **two backgrounds' logFC vectors agree**. A real, shared RT effect must reproduce
in both. This is scored against two references:

* a **permutation null** — shuffle the RT label among samples within each background. Necessary
  because 8,006 genes load on a few shared state axes, so the parametric p is meaningless.
* the **treated-vs-NoTx benchmark** — the same pipeline, same cells, same n, on a perturbation
  07 already showed is real. This is the r a genuine effect produces here.

In [8]:
def pseudobulk(mask):
    d = adata.obs.loc[mask]
    ok = d.groupby("sample", observed=True).size()
    ok = set(ok[ok >= MIN_CELLS].index)
    idx = np.where(mask & adata.obs["sample"].isin(ok).values)[0]
    sub, ot = adata.X[idx], adata.obs.iloc[idx]
    samples = list(ot["sample"].unique())
    pb = pd.DataFrame(np.vstack([np.asarray(sub[(ot["sample"]==s).values].mean(axis=0)).ravel()
                                 for s in samples]), index=samples, columns=adata.var_names)
    det = np.asarray((sub > 0).mean(axis=0)).ravel()
    return pb.loc[:, det > DET_FLOOR], ot.groupby("sample", observed=True)[["cond"]].first().loc[samples]

def concordance(pb, meta, arms_bg, nperm=2000):
    """centred cross-background logFC correlation + within-background label-permutation null"""
    def r_of(lab):
        v = {}
        for bg, (a_arm, b_arm) in arms_bg.items():
            arms = meta.index[meta.cond.isin([a_arm, b_arm])]
            g = np.array([lab[s] for s in arms])
            if g.sum() < 2 or (~g).sum() < 2: return np.nan
            d = pb.loc[arms[g]].mean(0) - pb.loc[arms[~g]].mean(0)
            v[bg] = d - d.mean()                      # centring kills the global offset
        return stats.pearsonr(v["PSCA"], v["TAG72"])[0]
    true = {s: meta.loc[s,"cond"] in [a for a,_ in arms_bg.values()] for s in meta.index}
    r = r_of(true)
    null = []
    for _ in range(nperm):
        lab = dict(true)
        for bg, (a_arm, b_arm) in arms_bg.items():
            arms = list(meta.index[meta.cond.isin([a_arm,b_arm])])
            vals = [true[s] for s in arms]; rng.shuffle(vals); lab.update(dict(zip(arms, vals)))
        rr = r_of(lab)
        if np.isfinite(rr): null.append(rr)
    null = np.array(null)
    return r, float(np.mean(np.abs(null) >= abs(r))), null

pb_tam, meta_tam = pseudobulk(adata.obs.TAM_global.values)
print(f"SPP1-TAM pseudobulk: {pb_tam.shape[0]} samples x {pb_tam.shape[1]:,} genes\n")

r_rt, p_rt, null = concordance(pb_tam, meta_tam, BG)
r_tx, p_tx, _    = concordance(pb_tam, meta_tam, {"PSCA":("CyPSCA","NoTx"), "TAG72":("CyTAG72","NoTx")})
print(f"  RT vs matched non-RT : r = {r_rt:+.3f}   permutation p = {p_rt:.3f}")
print(f"  permutation null     : mean {null.mean():+.3f}, sd {null.std():.3f}, "
      f"95% [{np.percentile(null,2.5):+.3f}, {np.percentile(null,97.5):+.3f}]")
print(f"  BENCHMARK Tx vs NoTx : r = {r_tx:+.3f}   permutation p = {p_tx:.3f}")
print(f"\n  => a genuinely shared perturbation reproduces at r = {r_tx:+.2f}.")
print(f"     The RT contrast sits inside its own null (p = {p_rt:.2f}) — no shared RT component.")

SPP1-TAM pseudobulk: 29 samples x 8,006 genes



  RT vs matched non-RT : r = -0.288   permutation p = 0.492
  permutation null     : mean +0.006, sd 0.359, 95% [-0.648, +0.695]
  BENCHMARK Tx vs NoTx : r = +0.888   permutation p = 0.004

  => a genuinely shared perturbation reproduces at r = +0.89.
     The RT contrast sits inside its own null (p = 0.49) — no shared RT component.


## 8b. The pseudobulk done **properly** — DESeq2 on summed raw counts

§8 averaged per-cell `log1p(CP10K)` per sample. That is convenient but **wrong for inference**: it
carries a per-cell library-*complexity* term, so a sample whose cells detect more genes shifts
every gene at once. That is the exact offset that faked a metabolic effect in §5.

The correct construction, and the one to quote:

1. **Sum raw counts** (`layers['counts']`) per sample within the population → a genes × samples
   integer matrix. Cells are aggregated, not averaged, so a sample is one observation.
2. **DESeq2** (negative binomial, median-of-ratios size factors, empirical-Bayes dispersion
   shrinkage). Size factors normalise depth *at the sample level* — the level the test runs at.
3. **`~ tu + cond`** — tumour site as a covariate, because Tu1 and Tu2 differ at baseline.

Both readouts are kept, because they answer different questions: DESeq2 gives correct per-gene
inference, while §8's permutation concordance is the non-parametric reproducibility test that
n = 3 mice actually supports and is cheap enough to sweep across every compartment in §9.

In [9]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

with h5py.File(SC_REF, "r") as f:
    Craw = ad.io.read_elem(f["layers/counts"])
C = sp.csr_matrix(Craw); del Craw
assert C.shape[0] == adata.n_obs
print(f"raw counts loaded: {C.shape[0]:,} x {C.shape[1]:,} | integer = "
      f"{float(C[:1000].data.max()) == int(C[:1000].data.max())}")

def pseudobulk_counts(mask, min_cells=MIN_CELLS):
    """SUM raw counts per sample — one observation per sample, not per cell."""
    d = adata.obs[mask]; n = d.groupby("sample", observed=True).size()
    ok = list(n[n >= min_cells].index)
    idx = np.where(mask & adata.obs["sample"].isin(ok).values)[0]
    sub, ot = C[idx], adata.obs.iloc[idx]
    M = np.vstack([np.asarray(sub[(ot["sample"]==s).values].sum(axis=0)).ravel() for s in ok])
    meta = ot.groupby("sample", observed=True)[["cond","tu","mouse"]].first().loc[ok]
    return pd.DataFrame(M.astype(np.int64), index=ok, columns=adata.var_names), meta

def run_deseq(cdf, meta, contrast, design="~ tu + cond", min_count=10, min_samples=3):
    keep = ((cdf > 0).sum(axis=0) >= min_samples) & (cdf.sum(axis=0) >= min_count*len(cdf))
    X = cdf.loc[:, keep]
    dds = DeseqDataSet(counts=X, metadata=meta.loc[X.index], design=design, quiet=True)
    dds.deseq2()
    st = DeseqStats(dds, contrast=contrast, quiet=True); st.summary()
    return st.results_df.sort_values("pvalue"), X.shape[1]

DE = {}
for pop, mk in [("TAM_global", adata.obs.TAM_global.values), ("M2_all", adata.obs.is_M2.values),
                ("mac_all", adata.obs.is_mac.values)]:
    cdf, meta = pseudobulk_counts(mk)
    print(f"\n=== {pop}: {mk.sum():,} cells -> {len(cdf)} sample pseudobulks, "
          f"median library {cdf.sum(axis=1).median():,.0f} counts ===")
    for bg, (rt, ct) in BG.items():                      # the RT contrast
        sel = meta.cond.isin([rt, ct])
        m = meta[sel].copy(); m["cond"] = pd.Categorical(m.cond, [ct, rt])
        res, ng = run_deseq(cdf[sel.values], m, ["cond", rt, ct]); DE[(pop,"RT",bg)] = res
        n = int((res.padj < 0.05).sum())
        print(f"  RT {bg:<6s}: {sel.sum():2d} samples, {ng:,} genes -> {n:>5,} at padj<0.05"
              + (f"   [{', '.join(res.index[:6])}]" if n else ""))
    for bg, arm in [("PSCA","CyPSCA"), ("TAG72","CyTAG72")]:   # the benchmark
        sel = meta.cond.isin([arm, "NoTx"])
        m = meta[sel].copy(); m["cond"] = pd.Categorical(m.cond, ["NoTx", arm])
        res, ng = run_deseq(cdf[sel.values], m, ["cond", arm, "NoTx"]); DE[(pop,"TX",bg)] = res
        print(f"  TX {bg:<6s}: {sel.sum():2d} samples, {ng:,} genes -> "
              f"{int((res.padj<0.05).sum()):>5,} at padj<0.05   <- same machinery, real effect")
for k, v in DE.items(): v.to_csv(OUT/f"deseq_{k[0]}_{k[1]}_{k[2]}.csv")

raw counts loaded: 93,548 x 22,949 | integer = True

=== TAM_global: 12,119 cells -> 29 sample pseudobulks, median library 4,446,923 counts ===


  RT PSCA  : 12 samples, 10,238 genes ->     0 at padj<0.05


  RT TAG72 : 11 samples, 11,675 genes ->     3 at padj<0.05   [Erdr1, Mid1, Lyz1, Suv39h2, Klf8, Ccl2]


  TX PSCA  : 12 samples, 9,440 genes -> 2,683 at padj<0.05   <- same machinery, real effect


  TX TAG72 : 11 samples, 10,832 genes -> 4,237 at padj<0.05   <- same machinery, real effect


...


=== mac_all: 48,475 cells -> 29 sample pseudobulks, median library 17,937,054 counts ===


  RT PSCA  : 12 samples, 12,624 genes ->     0 at padj<0.05


  RT TAG72 : 11 samples, 13,499 genes ->     6 at padj<0.05   [Erdr1, Mill2, Cd300lh, Lyz1, Suv39h2, Mid1]


  TX PSCA  : 12 samples, 11,543 genes -> 4,356 at padj<0.05   <- same machinery, real effect


  TX TAG72 : 11 samples, 12,299 genes -> 5,206 at padj<0.05   <- same machinery, real effect


### 8c. The handful of "significant" RT genes are mouse identity, not radiation

The RT contrast returns 0 genes in PSCA and a few in TAG72. Before calling those a finding, apply
the test any real effect must pass: **does it replicate in the other CAR-T background?**

In [10]:
for pop in ["TAM_global","M2_all","mac_all"]:
    s = DE[(pop,"RT","TAG72")]; s = s[s.padj < 0.05]
    if not len(s): continue
    other = DE[(pop,"RT","PSCA")].reindex(s.index)[["log2FoldChange","padj"]]
    out = s[["baseMean","log2FoldChange","padj"]].join(other, rsuffix="_PSCA")
    print(f"\n--- {pop}: genes significant in the TAG72 RT contrast, and what PSCA says ---")
    print(out.to_string(float_format=lambda x: f"{x:.3g}"))
print("""
READ: Erdr1 (strain-polymorphic, notoriously variable in mouse scRNA-seq), Mid1 (pseudoautosomal,
sex-linked), Mill2 / Gm13710 / Rpl39l / Lyz1 / Suv39h2 / Cd300lh (strain-variable or predicted-gene
loci). Not one is a fatty-acid gene, they form no coherent program, and every one FLIPS SIGN and
loses significance in the other background. This is the signature of WHICH MICE landed in which
group -- n = 3 animals per arm -- not of radiation.""")


--- TAM_global: genes significant in the TAG72 RT contrast, and what PSCA says ---
       baseMean  log2FoldChange     padj  log2FoldChange_PSCA  padj_PSCA
Erdr1       986            1.98 0.000244               -0.348          1
Mid1        268            1.08 0.000317               -0.215          1
Lyz1        158           -1.16  0.00699               0.0615          1

--- M2_all: genes significant in the TAG72 RT contrast, and what PSCA says ---
        baseMean  log2FoldChange     padj  log2FoldChange_PSCA  padj_PSCA
Erdr1   2.05e+03            2.05 0.000244               -0.425      0.999
Rpl39l      12.8            3.29 0.000244                  NaN        NaN
Mid1         500             1.3  0.00509              -0.0107      0.999

--- mac_all: genes significant in the TAG72 RT contrast, and what PSCA says ---
         baseMean  log2FoldChange     padj  log2FoldChange_PSCA  padj_PSCA
Erdr1    4.19e+03            2.14 0.000709               -0.632          1
Mill2         108

### 8d. Pathway enrichment on the DESeq2 ranking, and concordance on the Wald statistic

The properly-powered version of the fatty-acid question: rank every gene by its signed DESeq2 Wald
statistic and ask whether the lipid/FA sets concentrate at the RT-up end — with the
treated-vs-NoTx benchmark scored the same way, in the same cells, at the same n.

In [11]:
def gsea_z(res, setgenes):
    r = res.dropna(subset=["stat"]); inset = r.index.isin(setgenes)
    if inset.sum() < 8: return np.nan, int(inset.sum())
    a, b = r.stat[inset], r.stat[~inset]
    auc = stats.mannwhitneyu(a, b).statistic/(len(a)*len(b))
    return (auc-0.5)/np.sqrt((len(a)+len(b)+1)/(12*len(a)*len(b))), int(inset.sum())

PANEL = {k: ALL_SETS[k] for k in ["FA_metabolism_H","OXPHOS","FA_beta_oxidation","FA_catabolism",
         "lipid_oxidation","FA_transport","FA_uptake_import","lipid_droplet","chol_efflux",
         "foam_cell","oxLDL_response","adipogenesis","peroxisome","lysosomal_transport",
         "glycolysis","hypoxia","IFN_gamma","TNFA_NFKB","inflammatory","MYC_targets",
         "p53_pathway","DNA_repair","apoptosis","ROS_pathway"]}
rows = []
for short, full in PANEL.items():
    g = gmt[full]
    zp,_ = gsea_z(DE[("TAM_global","RT","PSCA")], g); zt,n = gsea_z(DE[("TAM_global","RT","TAG72")], g)
    bp,_ = gsea_z(DE[("TAM_global","TX","PSCA")], g); bt,_ = gsea_z(DE[("TAM_global","TX","TAG72")], g)
    rows.append(dict(pathway=short, tier=TIER[short], n=n, RT_PSCA=zp, RT_TAG72=zt,
                     RT_agree="YES" if np.sign(zp)==np.sign(zt) else "no",
                     TX_PSCA=bp, TX_TAG72=bt, TX_agree="YES" if np.sign(bp)==np.sign(bt) else "no"))
E = pd.DataFrame(rows); E.to_csv(OUT/"deseq_pathway_enrichment.csv", index=False)
print(E.to_string(index=False, float_format=lambda x: f"{x:+.2f}"))
met = E[E.tier=="metabolic"]
print(f"\n  RT contrast : {(E.RT_agree=='YES').sum()}/{len(E)} sets agree in sign across backgrounds"
      f"  (chance ~{len(E)//2})")
print(f"  TX benchmark: {(E.TX_agree=='YES').sum()}/{len(E)} agree   <- what a real effect looks like")
print(f"  metabolic sets only: RT agreement {(met.RT_agree=='YES').sum()}/{len(met)}, "
      f"max |z| {met[['RT_PSCA','RT_TAG72']].abs().max().max():.2f}"
      f"   (benchmark max |z| {E[['TX_PSCA','TX_TAG72']].abs().max().max():.1f})")

print("\n--- cross-background concordance of the DESeq2 Wald statistic ---")
for pop in ["TAM_global","M2_all","mac_all"]:
    for kind, lbl in [("RT","RT vs matched non-RT"), ("TX","treated vs NoTx (benchmark)")]:
        a, b = DE[(pop,kind,"PSCA")], DE[(pop,kind,"TAG72")]
        c = a.index.intersection(b.index); x, y = a.loc[c,"stat"], b.loc[c,"stat"]
        ok = x.notna() & y.notna()
        print(f"  {pop:<12s} {lbl:<32s} r = {stats.pearsonr(x[ok], y[ok])[0]:+.3f}   ({ok.sum():,} genes)")
print("\n  => the count-based, size-factor-normalised analysis reproduces the §8 conclusion exactly.")

            pathway      tier   n  RT_PSCA  RT_TAG72 RT_agree  TX_PSCA  TX_TAG72 TX_agree
    FA_metabolism_H metabolic 119    -3.34     +0.37       no    -3.26     -3.73      YES
             OXPHOS metabolic 177    -0.28     -1.75      YES    -6.02     -7.37      YES
  FA_beta_oxidation metabolic  57    -1.11     +0.07       no    -1.03     -0.59      YES
      FA_catabolism metabolic  75    -1.50     +0.06       no    -0.33     -0.86      YES
    lipid_oxidation metabolic  85    -1.33     +0.00       no    -0.76     -0.13      YES
       FA_transport metabolic  62    -0.75     +0.25       no    +0.02     -0.75       no
   FA_uptake_import metabolic  14    -0.47     +0.67       no    -0.42     -0.97      YES
      lipid_droplet metabolic  28    +0.42     -0.97       no    -0.47     +0.15       no
        chol_efflux metabolic  39    -0.64     -0.75      YES    +2.20     +2.23      YES
          foam_cell metabolic  16    -0.62     +0.39       no    +0.66     +0.43      YES
     oxLDL

## 9. §E — Is radiation visible **anywhere** in this dataset?

This is the disambiguation that decides how the null is worded. *"RT does nothing to SPP1-TAMs"*
and *"this experiment cannot resolve RT"* are different claims with different consequences.

The same reproducibility test, run on every compartment, with treated-vs-NoTx carried alongside
as the positive control in each one.

In [12]:
POPS = {
  "SPP1-TAM (sig top q)": adata.obs.TAM_global.values, "M2_like_Mac": adata.obs.is_M2.values,
  "M1_like_Mac": (adata.obs.ct=="M1_like_Mac").values,
  "Intermediate_Mac": (adata.obs.ct=="Intermediate_Mac").values,
  "all macrophages": adata.obs.is_mac.values, "Classical_Mono": (adata.obs.ct=="Classical_Mono").values,
  "Cancer_cell": (adata.obs.ct=="Cancer_cell").values,
  "Neutrophils": adata.obs.ct.isin(["N1_like_Neu","N2_like_Neu"]).values,
  "cDC": (adata.obs.ct=="cDC").values, "ALL CELLS": np.ones(adata.n_obs, bool)}
TX = {"PSCA":("CyPSCA","NoTx"), "TAG72":("CyTAG72","NoTx")}

print(f"{'compartment':<24s} {'n cells':>9s} {'genes':>7s} | {'RT vs noRT':>11s} {'p':>7s} | {'Tx vs NoTx':>11s} {'p':>7s}")
print("-"*92)
rows=[]
for name, mk in POPS.items():
    try:
        pb, mt = pseudobulk(mk)
        r1,p1,_ = concordance(pb, mt, BG, nperm=1000)
        r2,p2,_ = concordance(pb, mt, TX, nperm=1000)
    except Exception:
        r1=p1=r2=p2=np.nan; pb=pd.DataFrame()
    print(f"{name:<24s} {mk.sum():>9,} {pb.shape[1]:>7,} | {r1:>+11.3f} {p1:>7.3f} | {r2:>+11.3f} {p2:>7.3f}")
    rows.append(dict(compartment=name, n_cells=int(mk.sum()), r_RT=r1, p_RT=p1, r_TX=r2, p_TX=p2))
pd.DataFrame(rows).to_csv(OUT/"rt_reproducibility_by_compartment.csv", index=False)
print("-"*92)
print("READ: 'Tx vs NoTx' is the positive control — identical pipeline, cells and n. It is high")
print("      EVERYWHERE. 'RT vs noRT' is at chance EVERYWHERE. The beam left no reproducible")
print("      transcriptional trace in any compartment, so the SPP1-TAM null is not TAM-specific.")

compartment                n cells   genes |  RT vs noRT       p |  Tx vs NoTx       p
--------------------------------------------------------------------------------------------


SPP1-TAM (sig top q)        12,119   8,006 |      -0.288   0.478 |      +0.888   0.008


M2_like_Mac                 23,538   7,861 |      -0.352   0.407 |      +0.855   0.016


M1_like_Mac                 15,716   7,020 |      -0.479   0.297 |      +0.799   0.074


Intermediate_Mac             9,221   7,720 |      -0.406   0.383 |      +0.811   0.048


...

Classical_Mono              25,955   6,333 |      -0.389   0.366 |      +0.856   0.027


Cancer_cell                  6,614   3,787 |      -0.086   0.882 |      +0.848   0.055


Neutrophils                  3,813   2,229 |      -0.173   0.507 |      +0.764   0.010


cDC                          1,381   6,710 |      +0.006   0.982 |      +0.867   0.001


ALL CELLS                   93,548   7,072 |      -0.401   0.394 |      +0.875   0.035
--------------------------------------------------------------------------------------------
READ: 'Tx vs NoTx' is the positive control — identical pipeline, cells and n. It is high
      EVERYWHERE. 'RT vs noRT' is at chance EVERYWHERE. The beam left no reproducible
      transcriptional trace in any compartment, so the SPP1-TAM null is not TAM-specific.


## 10. §F — The strongest contrast the design allows: irradiated vs abscopal, within mouse

`Tu1` and `Tu2` come from the **same animal** — same batch, same Cy + CAR-T, same day. Only the
beam differs. This removes the mouse-to-mouse variation that dominated every between-arm contrast.

Three things must all hold for a real local RT effect, and the **non-RT arms are the control for
every one of them**, because `Tu1 − Tu2` there is the baseline site difference with no radiation:
consistency across the mice of an arm, consistency between the two RT arms, and a larger
magnitude in RT arms than in control arms.

In [13]:
def mouse_deltas(arm):
    a = meta_tam[meta_tam.cond==arm].join(adata.obs.groupby("sample", observed=True)[["tu","mouse"]].first())
    out = {}
    for mouse, grp in a.groupby("mouse", observed=True):
        s1, s2 = grp.index[grp.tu=="Tu1"], grp.index[grp.tu=="Tu2"]
        if len(s1) and len(s2):
            d = pb_tam.loc[s1[0]] - pb_tam.loc[s2[0]]
            out[mouse] = d - d.mean()
    return pd.DataFrame(out)

D = {arm: mouse_deltas(arm) for arm in ["RTCyPSCA","RTCyTAG72","CyPSCA","CyTAG72","NoTx"]}

print("A. consistency ACROSS MICE within an arm (mean pairwise r of the per-mouse Tu1-Tu2 vectors)")
for arm, d in D.items():
    if d.shape[1] < 2: continue
    rs = [stats.pearsonr(d.iloc[:,i], d.iloc[:,j])[0]
          for i in range(d.shape[1]) for j in range(i+1, d.shape[1])]
    print(f"  {arm:<10s} {'IRRADIATED' if arm.startswith('RT') else 'control   '} "
          f"n={d.shape[1]}  mean r {np.mean(rs):+.3f}   [{', '.join(f'{r:+.2f}' for r in rs)}]")

armmean = lambda arm: D[arm].mean(axis=1) - D[arm].mean(axis=1).mean()
print("\nB. consistency BETWEEN the two arms of a pair")
print(f"  RT arms    (RTCyPSCA vs RTCyTAG72) r = {stats.pearsonr(armmean('RTCyPSCA'), armmean('RTCyTAG72'))[0]:+.3f}")
print(f"  non-RT ctl (CyPSCA   vs CyTAG72  ) r = {stats.pearsonr(armmean('CyPSCA'),   armmean('CyTAG72'))[0]:+.3f}")
print("  -> if the control pair is as consistent as the RT pair, the consistency is the SITE, not RT.")

print("\nC. MAGNITUDE of the per-mouse Tu1-Tu2 difference (RMS)")
for arm, d in D.items():
    if d.shape[1]==0: continue
    mags = np.sqrt((d**2).mean(axis=0))
    print(f"  {arm:<10s} {'IRRADIATED' if arm.startswith('RT') else 'control   '} "
          f"RMS {mags.mean():.4f}  [{', '.join(f'{v:.4f}' for v in mags)}]")

A. consistency ACROSS MICE within an arm (mean pairwise r of the per-mouse Tu1-Tu2 vectors)
  RTCyPSCA   IRRADIATED n=3  mean r +0.071   [-0.41, +0.17, +0.46]
  RTCyTAG72  IRRADIATED n=3  mean r +0.178   [+0.34, +0.36, -0.17]
  CyPSCA     control    n=3  mean r -0.039   [-0.27, -0.14, +0.29]
  CyTAG72    control    n=2  mean r -0.459   [-0.46]
  NoTx       control    n=3  mean r +0.262   [+0.16, +0.24, +0.38]

B. consistency BETWEEN the two arms of a pair
  RT arms    (RTCyPSCA vs RTCyTAG72) r = +0.539
  non-RT ctl (CyPSCA   vs CyTAG72  ) r = +0.618
  -> if the control pair is as consistent as the RT pair, the consistency is the SITE, not RT.

C. MAGNITUDE of the per-mouse Tu1-Tu2 difference (RMS)
  RTCyPSCA   IRRADIATED RMS 0.1469  [0.1765, 0.1697, 0.0947]
  RTCyTAG72  IRRADIATED RMS 0.0836  [0.0702, 0.0868, 0.0938]
  CyPSCA     control    RMS 0.1148  [0.0442, 0.1591, 0.1410]
  CyTAG72    control    RMS 0.0962  [0.1003, 0.0921]
  NoTx       control    RMS 0.0757  [0.0543, 0.0733, 0.09

## 11. SUMMARY — report these back

In [14]:
print("="*88)
print("SUMMARY — what does RT do to the SPP1-TAM state? (single-cell, annotation_v4)")
print("="*88)
print(f"""
DESIGN     29 samples / 15 mice / 5 arms x 2 tumour sites. No RT-alone arm: every contrast is
           'adding RT on top of Cy + CAR-T'. n = 3 mice (5-6 tumours) per side.
POPULATION SPP1-TAM defined 4 ways (signature global / within-sample / Alice's Spp1_pos_mac /
           M2_like_Mac). Every result below holds under all four.

1. FATTY ACID  -- NO INCREASE.  Internal-reference FA composite by arm:
     NoTx {rel[rel.cond=='NoTx'].FA_composite.mean():+.4f} | CyPSCA {rel[rel.cond=='CyPSCA'].FA_composite.mean():+.4f} | RTCyPSCA {rel[rel.cond=='RTCyPSCA'].FA_composite.mean():+.4f}"""
      f""" | CyTAG72 {rel[rel.cond=='CyTAG72'].FA_composite.mean():+.4f} | RTCyTAG72 {rel[rel.cond=='RTCyTAG72'].FA_composite.mean():+.4f}
   RT deltas are -0.003..+0.002 against a within-arm sd of {sd_within:.4f}.

2. POWER IS NOT THE PROBLEM.  The same readout separates Spp1-high from Spp1-low macrophages by
   {vs.mean():+.4f} (Cohen d 1.24), positive in {int((vs>0).sum())}/{len(vs)} samples. MDE = {mde:.4f}, i.e. ~{100*mde/vs.mean():.0f}% of the
   state axis. An RT effect a third the size of the SPP1-TAM state itself would have been seen.

3. THE CONTROLS FIRE.  In the naive between-arm contrast the biggest mover is TNFA_NFKB (a CONTROL
   pathway), opposite signs in the two backgrounds -> a per-sample global offset, not biology.
   After the internal reference and within-mouse pairing, the metabolic tier does not stand above
   the control tier.

4. NO SHARED RT EFFECT AT GENE LEVEL.  Cross-background logFC r = {r_rt:+.3f}
   (permutation p = {p_rt:.3f}) -- inside its own null. BENCHMARK, same pipeline:
   treated-vs-NoTx r = {r_tx:+.3f} (p = {p_tx:.3f}).

4b. PROPER PSEUDOBULK (section 8b: summed raw counts + DESeq2, ~ tu + cond) AGREES, more starkly:
   RT      -> 0 genes (PSCA) and 3-6 (TAG72) at padj<0.05; every TAG72 hit is a strain-polymorphic
              or sex-linked locus (Erdr1, Mid1, Mill2, Lyz1, ...) that FLIPS SIGN in PSCA.
   BENCH   -> 2,683-5,206 genes at padj<0.05, same machinery, same n.
   Wald-stat concordance: RT r = -0.13..-0.27  vs  benchmark r = +0.82..+0.88.
   Pathway sign-agreement across backgrounds: RT 5/24 (below chance), benchmark 18/24.

5. RT IS INVISIBLE IN EVERY COMPARTMENT (section 9), while Cy+CAR-T is strongly reproducible in
   every compartment. So this is NOT an SPP1-TAM-specific null.

6. WITHIN-MOUSE irradiated-vs-abscopal is null too, and the non-RT control pair is MORE internally
   consistent than the RT pair -- the Tu1/Tu2 difference is the tumour SITE, not the beam.

WHAT THIS LICENSES
   SAY : "adding RT on top of Cy + CAR-T did not detectably alter the fatty-acid / lipid-oxidative
          program of SPP1-TAMs; nor did it leave a reproducible transcriptional signature in any
          compartment, in a design powered to detect a third of the SPP1-TAM state effect."
   DO NOT SAY : "RT increases fatty acid processing in SPP1-TAMs." Nothing here supports it.
   NOTE : the SPP1-TAM lipid program itself is real and strong (01c, and control 1 above). What is
          absent is an effect OF RADIATION on it.

WHERE AN RT SIGNAL COULD STILL LIVE
   - SPATIAL ORGANISATION, not cell-intrinsic expression -> 08 + 08b: RUN, and this is where
     the one surviving RT finding is. RT does not change the STRENGTH of SPP1-TAM/niche
     coupling, but it extends its spatial RANGE (decay to ~37% of near-field by 100-150um
     without RT vs ~68-98% with it; background-matched, both CAR-T backgrounds agree).
     Systemic: abscopal tumours show it as strongly as irradiated. See 08b for the 4 controls.
   - An earlier timepoint. These are endpoint tumours; an acute RT response would have resolved.
""")
print("="*88)

SUMMARY — what does RT do to the SPP1-TAM state? (single-cell, annotation_v4)

DESIGN     29 samples / 15 mice / 5 arms x 2 tumour sites. No RT-alone arm: every contrast is
           'adding RT on top of Cy + CAR-T'. n = 3 mice (5-6 tumours) per side.
POPULATION SPP1-TAM defined 4 ways (signature global / within-sample / Alice's Spp1_pos_mac /
           M2_like_Mac). Every result below holds under all four.

1. FATTY ACID  -- NO INCREASE.  Internal-reference FA composite by arm:
     NoTx +0.0629 | CyPSCA +0.0568 | RTCyPSCA +0.0554 | CyTAG72 +0.0577 | RTCyTAG72 +0.0590
   RT deltas are -0.003..+0.002 against a within-arm sd of 0.0058.

2. POWER IS NOT THE PROBLEM.  The same readout separates Spp1-high from Spp1-low macrophages by
   +0.0259 (Cohen d 1.24), positive in 29/29 samples. MDE = 0.0093, i.e. ~36% of the
   state axis. An RT effect a third the size of the SPP1-TAM state itself would have been seen.

3. THE CONTROLS FIRE.  In the naive between-arm contrast the biggest mover i